Assignmnet no 3.Fine-tune GPT or GPT-2 for creative story generation.

**Install Library**

In [20]:
!pip install transformers datasets accelerate pandas --quiet


**Import Libraries**- **torch** → deep learning engine  
- **Dataset** → structured data holder  
- **GPT2Tokenizer** → text → tokens  
- **GPT2LMHeadModel** → GPT‑2 generator  
- **Trainer** → training loop manager  
- **TrainingArguments** → training config  
- **DataCollatorForLanguageModeling** → batch prep & padding


In [21]:
import torch
from datasets import Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)

**Read the dataset**

In [22]:
import pandas as pd
df=pd.read_csv("/content/stories.csv")

**To understand dataset**

In [23]:
print(df.shape)
print(df.columns)
print(df.head())

(1000, 4)
Index(['id', 'title', 'story', 'genre'], dtype='object')
       id                               title  \
0  457580   The Chronicles of the Cosmic Rift   
1  297904        Eldoria's Enchanted Whispers   
2  620436         Echoes of Whispered Shadows   
3  634687  Emerald Amulet Chronicles Revealed   
4  513427        The Shadows of St. Augustine   

                                               story                 genre  
0  In the year 2250, Earth had made significant s...       Science Fiction  
1  In a land far away, where the sun shone bright...               Fantasy  
2  Once upon a time, in a small, tranquil town ca...               Mystery  
3  Once upon a time in the 16th century, a small ...  Historical Adventure  
4  In the sun-drenched coastal city of St. August...              Thriller  


**Only use those useful columns**

In [24]:
df = df.dropna()
df = df.reset_index(drop=True)
df = df[['title','story','genre']]


**Convert columns(Title,Genre,story) into most readable forms**

In [25]:
df["text"] = df.apply(
    lambda x: f"Title: {x['title']}\nGenre: {x['genre']}\nStory: {x['story']}",
    axis=1
)
print(df["text"][0])


Title: The Chronicles of the Cosmic Rift
Genre: Science Fiction
Story: In the year 2250, Earth had made significant strides in space exploration and interstellar travel. The United Earth Government (UEG) had established colonies on Mars, Jupiter's moon Europa, and Saturn's moon Titan. The advancements in technology and science had led to the creation of the Cosmic Rift Exploration Agency (CREA), a government-funded organization tasked with exploring the unknown regions of space and discovering new worlds and resources.

    Dr. Amelia Hart, a brilliant astrophysicist, was the lead scientist at CREA's headquarters on Luna. She had devoted her entire life to understanding the mysteries of the universe and had become a pioneer in her field. She was determined to uncover the secrets of the cosmic rifts, a series of mysterious and seemingly unconnected energy anomalies that had started appearing throughout the galaxy.

    Dr. Hart assembled a diverse team of experts for her next mission, i

**splits a long text(stories) into smaller chunks of up to 400 words each and then return list of all chunks. **


In [26]:
def split_into_chunks(text, chunk_size=400):
    words = text.split()
    chunks = []

    for i in range(0, len(words), chunk_size):
        chunk = " ".join(words[i:i+chunk_size])
        chunks.append(chunk)

    return chunks


**Takes every row’s text, breaks it into manageable chunks, collects them all, and prints the total number of chunks for training**


In [27]:
all_chunks = []

for text in df["text"]:
    chunks = split_into_chunks(text)
    all_chunks.extend(chunks)

print("Total training chunks:", len(all_chunks))


Total training chunks: 2980


**Turning list of text chunks into a structured dataset object that can be used for training or preprocessing with Hugging Face tools.**


In [28]:
dataset = Dataset.from_dict({"text": all_chunks})


In [29]:
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")

tokenizer.pad_token = tokenizer.eos_token


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


**Converts all text chunks into fixed‑length token sequences (IDs) so the GPT‑2 model can train on them.**


In [30]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

tokenized_dataset = dataset.map(tokenize, batched=True)


Map:   0%|          | 0/2980 [00:00<?, ? examples/s]

**Data Collator(creating a data collator that dynamically pads sequences and formats them correctly for GPT‑2’s next‑word prediction training.)**

In [31]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


**Training Configuration(configures the training process—where to save, how long to train, batch sizes, logging, checkpointing, and whether to use GPU acceleration.)**

In [32]:
training_args = TrainingArguments(
    output_dir="./story_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    logging_steps=50,
    save_steps=500,
    fp16=torch.cuda.is_available()
)


**Trainer Setup(sets up a Trainer object that ties together the model, dataset, training settings, and data preparation so you can run and fine-tune)**

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)
trainer.train()


Step,Training Loss
50,2.360211
100,2.195995
150,2.134029
200,2.094624
250,2.084042
300,2.056280
350,2.045999
400,1.965501
450,1.927138
500,1.933201


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1119, training_loss=1.9586561030847245, metrics={'train_runtime': 794.3407, 'train_samples_per_second': 11.255, 'train_steps_per_second': 1.409, 'total_flos': 2335950766080000.0, 'train_loss': 1.9586561030847245, 'epoch': 3.0})

**Save model**

In [34]:
trainer.save_model("fine_tuned_story_gpt2")
tokenizer.save_pretrained("fine_tuned_story_gpt2")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('fine_tuned_story_gpt2/tokenizer_config.json',
 'fine_tuned_story_gpt2/tokenizer.json')

**Load Fine-Tuned Model + Tokenizer**

In [35]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch

# Path where your fine-tuned model is saved
model_path = "fine_tuned_story_gpt2"

# Load tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained(model_path)
model = GPT2LMHeadModel.from_pretrained(model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

model.eval()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

**Take Prompt From User,Encode prompt and generate story**

In [36]:
# Take input from user
user_prompt = input("Enter story prompt: ")
inputs = tokenizer.encode(user_prompt, return_tensors="pt").to(device)
with torch.no_grad():
    outputs = model.generate(
        inputs,
        max_length=300,          # total story length
        temperature=0.9,         # creativity control (0.7–1.0 good)
        top_k=50,                # sampling diversity
        top_p=0.95,              # nucleus sampling
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )
generated_story = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("\nGenerated Story:\n")
print(generated_story)


Enter story prompt: A mysterious door appeared in the forest

Generated Story:

A mysterious door appeared in the forest, where a large, ancient tree had been stowed away. The tree, which was said to possess the power to grant its possessor a wish, was said to possess the ability to create new forms of existence, as well as the ability to manipulate the physical world. As the tree grew stronger, the trees began to awaken, and the villagers grew to love and trust in each other. The tree, which had once threatened to destroy the village, was soon restored to its former glory, and the villagers celebrated their victory. The villagers, now a part of the village, grew to love and trust in one another, and soon they were hailed as heroes in their own right. One day, a curious fox named Lysander woke up, his eyes filled with curiosity and anticipation. Lysander was a fox who had been searching for a hidden treasure, but had never found it. He had been searching for clues about the treasure an

**Saved story in txt file with paras**

In [42]:
import textwrap

def format_story(text, width=80):
    wrapped = textwrap.fill(text, width=width)
    paragraphs = wrapped.split("\n")
    return "\n\n".join(paragraphs)


In [43]:
formatted_story = format_story(generated_story)

with open("generated_story.txt", "w", encoding="utf-8") as f:
    f.write(formatted_story)

print("Saved successfully")


Saved successfully
